# Project 2: Route Traffic, Demand & Operational Performance

Exploratory analysis of United Airlines route network — load factors, seasonality, OTP, and KPIs.

Run `python analysis/pipeline.py --all --csv` first to generate the output CSVs.

In [1]:
import sys
from pathlib import Path

# Ensure analysis modules are importable
_ANALYSIS = Path.cwd().parent / 'analysis' if Path.cwd().name == 'notebooks' else Path.cwd() / 'analysis'
_OUTPUTS  = Path.cwd().parent / 'outputs'  if Path.cwd().name == 'notebooks' else Path.cwd() / 'outputs'

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap

# ── Style ─────────────────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted')
UNITED_BLUE  = '#005DAA'
UNITED_GOLD  = '#FFC72C'
ACCENT_RED   = '#D62728'
FIG_SIZE_W   = (14, 5)
FIG_SIZE_SQ  = (10, 8)

pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', '{:.4f}'.format)
print('Imports OK. Looking for outputs in:', _OUTPUTS)


Imports OK. Looking for outputs in: /Users/sucheen/Documents/airlineRevenueOptimization/Project 2/outputs


In [2]:
# ── Load data ─────────────────────────────────────────────────────────────────
def load(filename, **kwargs):
    p = _OUTPUTS / filename
    if p.exists():
        return pd.read_csv(p, **kwargs)
    print(f'  WARNING: {filename} not found — run pipeline.py --all first.')
    return pd.DataFrame()

demand_df   = load('route_demand_summary.csv',  parse_dates=['report_period'])
lf_feed_df  = load('load_factor_feed.csv',       parse_dates=['report_period'])
perf_df     = load('route_operational_perf.csv', parse_dates=['report_period'])
season_df   = load('route_seasonality.csv')
kpi_df      = load('route_kpi_feed.csv',         parse_dates=['report_period'])
otp_sum_df  = load('aircraft_otp_summary.csv')
top_routes  = load('top_routes_by_demand.csv')
trends_df   = load('demand_trends.csv',          parse_dates=['report_period'])

print('Demand rows:',     len(demand_df))
print('Perf rows:',       len(perf_df))
print('Season rows:',     len(season_df))
print('KPI feed rows:',   len(kpi_df))


Demand rows: 0
Perf rows: 0
Season rows: 0
KPI feed rows: 0


## 1. Load Factor Distribution

In [3]:
if not demand_df.empty:
    lf = demand_df['load_factor_imputed'].dropna()

    fig, axes = plt.subplots(1, 2, figsize=FIG_SIZE_W)

    # Histogram
    axes[0].hist(lf, bins=40, color=UNITED_BLUE, edgecolor='white', alpha=0.85)
    axes[0].axvline(lf.mean(),   color=UNITED_GOLD, lw=2, linestyle='--', label=f'Mean {lf.mean():.3f}')
    axes[0].axvline(lf.median(), color=ACCENT_RED,  lw=2, linestyle=':',  label=f'Median {lf.median():.3f}')
    axes[0].set_xlabel('Load Factor')
    axes[0].set_ylabel('Frequency')
    axes[0].set_title('UA Route Load Factor Distribution')
    axes[0].legend()

    # Boxplot by year
    demand_df['year'] = pd.to_datetime(demand_df['report_period']).dt.year
    axes[1].boxplot(
        [demand_df.loc[demand_df['year'] == y, 'load_factor_imputed'].dropna()
         for y in sorted(demand_df['year'].unique())],
        labels=sorted(demand_df['year'].unique()),
        patch_artist=True,
        boxprops=dict(facecolor=UNITED_BLUE, alpha=0.7),
        medianprops=dict(color=UNITED_GOLD, lw=2),
    )
    axes[1].set_xlabel('Year')
    axes[1].set_ylabel('Load Factor')
    axes[1].set_title('UA Load Factor by Year (Boxplot)')

    plt.tight_layout()
    plt.savefig(_OUTPUTS / 'plot_lf_distribution.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'LF stats — mean: {lf.mean():.3f}  median: {lf.median():.3f}  '
          f'std: {lf.std():.3f}  p10: {lf.quantile(0.1):.3f}  p90: {lf.quantile(0.9):.3f}')
else:
    print('No demand data loaded.')


No demand data loaded.


## 2. Load Factor Imputation Quality

In [4]:
if not lf_feed_df.empty and 'imputation_method' in lf_feed_df.columns:
    method_counts = lf_feed_df['imputation_method'].value_counts()
    fig, ax = plt.subplots(figsize=(8, 4))
    method_counts.plot(kind='barh', ax=ax, color=UNITED_BLUE, edgecolor='white')
    ax.set_xlabel('Number of Route-Period Rows')
    ax.set_title('Load Factor Imputation Method Breakdown')
    plt.tight_layout()
    plt.savefig(_OUTPUTS / 'plot_lf_imputation.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(method_counts.to_string())
    n_direct = (lf_feed_df['imputation_method'] == 'direct').sum()
    print(f'\nDirect (T-100): {n_direct}/{len(lf_feed_df)} ({n_direct/len(lf_feed_df):.1%})')


## 3. Seasonality — Monthly Load Factor Index

In [5]:
if not season_df.empty:
    # Top 20 routes by avg monthly pax
    top_routes_list = (
        demand_df.groupby(['origin_iata', 'destination_iata'])['passengers']
        .mean().nlargest(20).index.tolist()
        if not demand_df.empty else []
    )

    if top_routes_list:
        season_df['route'] = season_df['origin_iata'] + '-' + season_df['destination_iata']
        top_route_strs = [f'{o}-{d}' for o, d in top_routes_list]
        plot_df = season_df[season_df['route'].isin(top_route_strs)]
    else:
        season_df['route'] = season_df['origin_iata'] + '-' + season_df['destination_iata']
        top_routes_by_lf = season_df.groupby('route')['seasonality_index_lf'].mean().nlargest(20).index
        plot_df = season_df[season_df['route'].isin(top_routes_by_lf)]

    if not plot_df.empty:
        pivot = plot_df.pivot_table(
            index='route', columns='month_of_year',
            values='seasonality_index_lf', aggfunc='mean'
        )
        pivot.columns = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

        fig, ax = plt.subplots(figsize=(14, max(6, len(pivot) * 0.4)))
        cmap = LinearSegmentedColormap.from_list('ua_heat', ['#003580', '#FFFFFF', '#D62728'])
        sns.heatmap(pivot, ax=ax, cmap=cmap, center=1.0, vmin=0.5, vmax=1.5,
                    annot=True, fmt='.2f', linewidths=0.5, cbar_kws={'label': 'Seasonality Index (LF)'})
        ax.set_title('Monthly Seasonality Index (Load Factor) — Top UA Routes\n'
                     'Values > 1.0 = above-average month; < 1.0 = below-average', pad=12)
        ax.set_xlabel('Month')
        ax.set_ylabel('Route')
        plt.tight_layout()
        plt.savefig(_OUTPUTS / 'plot_seasonality_heatmap.png', dpi=150, bbox_inches='tight')
        plt.show()


## 4. On-Time Performance by Aircraft Type

In [6]:
if not otp_sum_df.empty:
    df_plot = otp_sum_df.dropna(subset=['avg_otp_pct']).sort_values('avg_otp_pct')

    fig, axes = plt.subplots(1, 2, figsize=FIG_SIZE_W)

    # OTP %
    colors = [UNITED_BLUE if v >= 0.80 else ACCENT_RED for v in df_plot['avg_otp_pct']]
    axes[0].barh(df_plot['aircraft_variant'], df_plot['avg_otp_pct'] * 100, color=colors, edgecolor='white')
    axes[0].axvline(80, color='black', lw=1, linestyle='--', label='80% threshold')
    axes[0].set_xlabel('OTP % (≤14 min delay)')
    axes[0].set_title('On-Time Performance by Aircraft Type')
    axes[0].legend()

    # Average arrival delay
    axes[1].barh(df_plot['aircraft_variant'], df_plot['avg_arrival_delay'],
                 color=UNITED_GOLD, edgecolor='white')
    axes[1].set_xlabel('Avg Arrival Delay (min)')
    axes[1].set_title('Average Arrival Delay by Aircraft Type')

    plt.tight_layout()
    plt.savefig(_OUTPUTS / 'plot_otp_by_aircraft.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(df_plot[['aircraft_variant','avg_otp_pct','avg_arrival_delay',
                   'avg_cancel_rate']].to_string(index=False))
elif not perf_df.empty:
    import operational as op_mod_nb
    otp = op_mod_nb.aircraft_otp_comparison(perf_df)
    print(otp.to_string(index=False))
else:
    print('No OTP data loaded.')


No OTP data loaded.


## 5. Schedule Padding (Scheduled vs Actual Block Time)

In [7]:
if not perf_df.empty and 'block_time_padding_min' in perf_df.columns:
    padding = (
        perf_df.groupby('aircraft_variant')['block_time_padding_min']
        .mean()
        .sort_values(ascending=False)
        .dropna()
    )

    fig, ax = plt.subplots(figsize=(10, 5))
    padding.plot(kind='bar', ax=ax, color=UNITED_BLUE, edgecolor='white')
    ax.axhline(0, color='black', lw=1)
    ax.set_ylabel('Avg Block Time Padding (min)')
    ax.set_xlabel('Aircraft Variant')
    ax.set_title('Schedule Padding: Scheduled − Actual Block Time\n'
                 'Positive = buffer built into schedule; Negative = chronically late')
    ax.tick_params(axis='x', rotation=45)
    plt.tight_layout()
    plt.savefig(_OUTPUTS / 'plot_block_time_padding.png', dpi=150, bbox_inches='tight')
    plt.show()


## 6. Route KPI Distributions

In [8]:
if not kpi_df.empty:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Load factor
    axes[0].hist(kpi_df['load_factor'].dropna(), bins=40, color=UNITED_BLUE,
                 edgecolor='white', alpha=0.85)
    axes[0].set_title('Load Factor Distribution')
    axes[0].set_xlabel('Load Factor')

    # Operational profit per block hour
    if 'op_profit_per_bh_usd' in kpi_df.columns:
        profit = kpi_df['op_profit_per_bh_usd'].dropna()
        axes[1].hist(profit, bins=40, color=UNITED_GOLD, edgecolor='white', alpha=0.85)
        axes[1].axvline(0, color=ACCENT_RED, lw=1.5, linestyle='--')
        axes[1].set_title('Operational Profit per Block Hour ($)')
        axes[1].set_xlabel('USD / Block Hour')

    # Estimated revenue
    if 'est_revenue_usd' in kpi_df.columns:
        rev = kpi_df['est_revenue_usd'].dropna() / 1e6
        axes[2].hist(rev, bins=40, color='#2CA02C', edgecolor='white', alpha=0.85)
        axes[2].set_title('Estimated Monthly Revenue ($M)')
        axes[2].set_xlabel('Revenue ($M / month per route)')

    plt.tight_layout()
    plt.savefig(_OUTPUTS / 'plot_kpi_distributions.png', dpi=150, bbox_inches='tight')
    plt.show()


## 7. Top Routes by Demand and Profitability

In [9]:
if not kpi_df.empty:
    # Aggregate to route-level
    route_agg = (
        kpi_df.groupby(['origin_iata', 'destination_iata', 'aircraft_variant'])
        .agg(
            avg_lf             =('load_factor',          'mean'),
            avg_op_profit_bh   =('op_profit_per_bh_usd', 'mean'),
            avg_pax            =('passengers_est',        'mean'),
            avg_otp            =('otp_pct',               'mean'),
            n_months           =('report_period',         'count'),
        )
        .reset_index()
        .sort_values('avg_op_profit_bh', ascending=False)
    )

    top20 = route_agg.head(20).copy()
    top20['route'] = top20['origin_iata'] + '→' + top20['destination_iata']
    top20['label'] = top20['route'] + ' (' + top20['aircraft_variant'].str[:6] + ')'

    fig, ax = plt.subplots(figsize=(14, 8))
    scatter = ax.scatter(
        top20['avg_lf'],
        top20['avg_op_profit_bh'],
        s=top20['avg_pax'].fillna(1000) / 20,
        c=top20['avg_otp'].fillna(0.8),
        cmap='RdYlGn', vmin=0.6, vmax=1.0,
        alpha=0.8, edgecolors='white', linewidths=0.5,
    )
    for _, row in top20.iterrows():
        ax.annotate(row['label'],
                    (row['avg_lf'], row['avg_op_profit_bh']),
                    fontsize=7, alpha=0.8,
                    xytext=(4, 4), textcoords='offset points')

    plt.colorbar(scatter, ax=ax, label='OTP %')
    ax.set_xlabel('Average Load Factor')
    ax.set_ylabel('Avg Operational Profit per Block Hour ($)')
    ax.set_title('Route Profitability: Load Factor vs Op Profit per BH\n'
                 '(bubble size = avg monthly passengers; color = OTP)')
    ax.axhline(0, color='black', lw=1, linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.savefig(_OUTPUTS / 'plot_route_profitability_scatter.png', dpi=150, bbox_inches='tight')
    plt.show()

    print('\n── Top 10 Routes by Operational Profit per Block Hour ──')
    print(top20.head(10)[['route','aircraft_variant','avg_lf',
                           'avg_op_profit_bh','avg_pax','avg_otp']].to_string(index=False))


## 8. Monthly Traffic Trend — Network-Level

In [10]:
if not demand_df.empty and 'passengers' in demand_df.columns:
    monthly = (
        demand_df.groupby('report_period')
        .agg(total_pax=('passengers','sum'),
             avg_lf=('load_factor_imputed','mean'),
             total_seats=('seats_available','sum'))
        .reset_index()
        .sort_values('report_period')
    )

    fig, ax1 = plt.subplots(figsize=FIG_SIZE_W)
    ax2 = ax1.twinx()

    ax1.fill_between(monthly['report_period'], monthly['total_pax'] / 1e6,
                     alpha=0.3, color=UNITED_BLUE)
    ax1.plot(monthly['report_period'], monthly['total_pax'] / 1e6,
             color=UNITED_BLUE, lw=2, label='Total Pax (M)')
    ax2.plot(monthly['report_period'], monthly['avg_lf'],
             color=UNITED_GOLD, lw=2, linestyle='--', label='Avg Load Factor')

    ax1.set_ylabel('Total Passengers (millions)', color=UNITED_BLUE)
    ax2.set_ylabel('Average Load Factor', color=UNITED_GOLD)
    ax1.set_title('UA Monthly Network Traffic & Load Factor')
    ax1.xaxis.set_major_locator(mticker.MaxNLocator(12))
    plt.xticks(rotation=30)

    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc='lower right')

    plt.tight_layout()
    plt.savefig(_OUTPUTS / 'plot_monthly_traffic.png', dpi=150, bbox_inches='tight')
    plt.show()


## 9. KPI Summary Table (Project 4 Input Feed Preview)

In [11]:
if not kpi_df.empty:
    print('KPI feed columns:', list(kpi_df.columns))
    print(f'\nShape: {kpi_df.shape}')
    print(f'Date range: {kpi_df.report_period.min()} → {kpi_df.report_period.max()}')
    print(f'Unique routes: {kpi_df.groupby(["origin_iata","destination_iata"]).ngroups}')
    print(f'Unique aircraft variants: {kpi_df.aircraft_variant.nunique()}')
    print()
    print('Sample rows:')
    display(kpi_df.sample(min(5, len(kpi_df))).T)
